# Thêm Thư Viện

In [4]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [5]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ CSV

In [6]:
df_data_date = pd.read_csv("./data_date.csv")
print(df_data_date)

       Date_key   Full_date                     Date_text   Day_name  \
0      19700101    1/1/1970     Thursday, January 1, 1970   Thursday   
1      19700102    1/2/1970       Friday, January 2, 1970     Friday   
2      19700103    1/3/1970     Saturday, January 3, 1970   Saturday   
3      19700104    1/4/1970       Sunday, January 4, 1970     Sunday   
4      19700105    1/5/1970       Monday, January 5, 1970     Monday   
...         ...         ...                           ...        ...   
29215  20491227  12/27/2049     Monday, December 27, 2049     Monday   
29216  20491228  12/28/2049    Tuesday, December 28, 2049    Tuesday   
29217  20491229  12/29/2049  Wednesday, December 29, 2049  Wednesday   
29218  20491230  12/30/2049   Thursday, December 30, 2049   Thursday   
29219  20491231  12/31/2049     Friday, December 31, 2049     Friday   

       Week_of_quarter  Day_of_week  Month  Quarter  Year  Day  
0                    1            4      1        1  1970    1  
1    

In [7]:
# Tạo hàng dữ liệu giả lập cho ngày không xác định
new_row = pd.DataFrame({
    'Date_key': [0],
    'Full_date': ['1/1/9999'],     # Đặt ngày giả định là 1900-01-01
    'Date_text': ['Unknown Date'],
    'Day': [0],
    'Week_of_quarter': [0],
    'Month': [0],
    'Quarter': [0],
    'Year': [9999],
    'Day_of_week': [0],
    'Day_name': ['Unknown']
})
df_data_date = pd.concat([df_data_date, new_row], ignore_index=True) # Thêm vào dataset
df_data_date = df_data_date.sort_values(by='Date_key', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
print(df_data_date)

       Date_key   Full_date                     Date_text   Day_name  \
0             0    1/1/9999                  Unknown Date    Unknown   
1      19700101    1/1/1970     Thursday, January 1, 1970   Thursday   
2      19700102    1/2/1970       Friday, January 2, 1970     Friday   
3      19700103    1/3/1970     Saturday, January 3, 1970   Saturday   
4      19700104    1/4/1970       Sunday, January 4, 1970     Sunday   
...         ...         ...                           ...        ...   
29216  20491227  12/27/2049     Monday, December 27, 2049     Monday   
29217  20491228  12/28/2049    Tuesday, December 28, 2049    Tuesday   
29218  20491229  12/29/2049  Wednesday, December 29, 2049  Wednesday   
29219  20491230  12/30/2049   Thursday, December 30, 2049   Thursday   
29220  20491231  12/31/2049     Friday, December 31, 2049     Friday   

       Week_of_quarter  Day_of_week  Month  Quarter  Year  Day  
0                    0            0      0        0  9999    0  
1    

## Load data

### [Nếu cần] Clear bảng 

In [8]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Date"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load vào bảng DIM_Date

In [9]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Date (Date_key, Full_date, Date_text, Day, Week_of_quarter, Month, Quarter, Year, Day_of_week, Day_name)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """
for index, row in df_data_date.iterrows():
    values = (row['Date_key'], 
              row['Full_date'], 
              row['Date_text'], 
              row['Day'], 
              row['Week_of_quarter'], 
              row['Month'], 
              row['Quarter'], 
              row['Year'], 
              row['Day_of_week'], 
              row['Day_name'])
    cursor_dwh.execute(insert_query, values)
    conn_dwh_library.commit()